# V15 — BENIGN-only Autoencoder ensemble training with port-specific calibration
Uses destination port as metadata for thresholds, not as a model feature.


**CICIDS2017 Improved:** numeric features are learned from `normal_train.csv`. No original CICIDS2017 feature names are hard-coded. Your processed CSV has no `Dst Port`, so the global validation threshold is used; local thresholds are disabled. Training and testing share `autoencoder_v9_common.py`.

## 1. Imports, paths and fixed configuration

In [1]:
from pathlib import Path
import json, random
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))
# Import helper after adding project root to sys.path.
from autoencoder_v15_common import (VERSION, FEATURES, LABELS, validate_frame, fit_preprocessing, transform,
                                   reconstruction_mse, benign_threshold)

DATA_DIR = PROJECT_ROOT / 'data' / 'processed' / 'unknown_detection'
TRAIN_FILE = DATA_DIR / 'normal_train.csv'
VAL_FILE = DATA_DIR / 'normal_validation.csv'
MODEL_DIR = PROJECT_ROOT / 'saved-models' / f'autoencoder_{VERSION}'
RESULTS_DIR = PROJECT_ROOT / 'outputs' / 'results' / f'autoencoder_{VERSION}'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Fixed configuration (decided before looking at any attack result) ----
HIDDEN = (48, 24)
LATENT = 12
ENSEMBLE_SEEDS = [42, 43, 44]
FPR_OPTIONS = [0.01, 0.03, 0.05]
TARGET_FPR = 0.03             # threshold = 97th percentile of benign validation errors
EPOCHS = 100; BATCH_SIZE = 1024; PRED_BATCH_SIZE = 4096
QUICK_RUN = False             # True = tiny smoke test
if QUICK_RUN: ENSEMBLE_SEEDS = [42, 43]; EPOCHS = 3
assert TARGET_FPR in FPR_OPTIONS
for p in (TRAIN_FILE, VAL_FILE):
    if not p.exists(): raise FileNotFoundError(p.resolve())
print('Version:', VERSION, '| labels:', LABELS, '| config:', HIDDEN, LATENT, 'seeds', ENSEMBLE_SEEDS)

Version: v15 | labels: ('BENIGN', 'Botnet', 'FTP-Patator', 'DoS Slowhttptest') | config: (48, 24) 12 seeds [42, 43, 44]


## 2. Load benign training and validation data (benign-only is enforced)

In [2]:
train_df, X_train = validate_frame(pd.read_csv(TRAIN_FILE, low_memory=False), 'train', benign_only=True)
val_df, X_val = validate_frame(pd.read_csv(VAL_FILE, low_memory=False), 'validation', benign_only=True)
print('Train:', X_train.shape, 'Validation:', X_val.shape, '(all BENIGN)')
prep = fit_preprocessing(X_train)
xtr = transform(X_train, prep); xval = transform(X_val, prep)
print('Model input dimension:', xtr.shape[1])

Train: (1002623, 74) Validation: (214848, 74) (all BENIGN)
Model input dimension: 73


## 3. Train the normal-only autoencoder ensemble

In [3]:
def make_model(input_dim, hidden, latent):
    inputs = Input(shape=(input_dim,))
    x = inputs
    for units in hidden: x = Dense(units, activation='relu')(x)
    x = Dense(latent, activation='linear', name='bottleneck')(x)
    for units in reversed(hidden): x = Dense(units, activation='relu')(x)
    outputs = Dense(input_dim, activation='linear')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
    return model

models, histories = [], []
for seed in ENSEMBLE_SEEDS:
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    model = make_model(xtr.shape[1], HIDDEN, LATENT)
    history = model.fit(xtr, xtr, validation_data=(xval, xval), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0,
        callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
                   ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6)])
    models.append(model); histories.append(history.history)
    print('seed', seed, '| epochs', len(history.history['loss']), '| best val loss %.5f' % min(history.history['val_loss']))


seed 42 | epochs 100 | best val loss 0.00360
seed 43 | epochs 100 | best val loss 0.00277
seed 44 | epochs 100 | best val loss 0.00264


## 4. Benign-only threshold

Scores are the seed-averaged reconstruction MSE of the benign validation set. The threshold is its `1 - FPR` percentile, so about `TARGET_FPR` of unseen normal traffic will be labelled UNKNOWN by design.

## 4. Global, per-port and service-group calibration
Minimum validation count is set to 500 for per-port thresholds and 1000 for service groups. Small groups fall back to global. Group thresholds are exploratory, not guaranteed to achieve exactly 3% FPR on future traffic.


In [4]:
val_scores = reconstruction_mse(models, xval, PRED_BATCH_SIZE)
TARGET_FPR = 0.03
GLOBAL_THRESHOLD = benign_threshold(val_scores, TARGET_FPR)
MIN_PORT_VALIDATION = 500
MIN_GROUP_VALIDATION = 1000

def service_group(port):
    port = int(port)
    if port in (20, 21): return 'ftp'
    if port in (80, 443, 8080, 8443): return 'web'
    if port == 22: return 'ssh'
    if port == 53: return 'dns'
    if port < 1024: return 'other_well_known'
    if port < 49152: return 'registered'
    return 'dynamic'

port_col = next((c for c in ('Destination Port','Dst Port') if c in val_df.columns), None)
if port_col is None:
    raise ValueError('Dst Port is missing. Re-run V15 Notebook 01 to preserve port metadata.')
val_ports = (pd.to_numeric(val_df[port_col],errors='coerce').to_numpy()
             if port_col else np.zeros(len(val_df), dtype=int))
val_groups = np.array([service_group(p) if pd.notna(p) and 0 <= p <= 65535 else 'unknown' for p in val_ports])
port_thresholds = {}
port_counts = {}
for port, count in zip(*np.unique(val_ports, return_counts=True)):
    if pd.isna(port) or not (0 <= port <= 65535): continue
    port_counts[str(int(port))] = int(count)
    if port_col is not None and count >= MIN_PORT_VALIDATION:
        port_thresholds[str(int(port))] = float(benign_threshold(val_scores[val_ports == port], TARGET_FPR))
group_thresholds = {}
group_counts = {}
for group, count in zip(*np.unique(val_groups, return_counts=True)):
    group_counts[str(group)] = int(count)
    if group != 'unknown' and count >= MIN_GROUP_VALIDATION:
        group_thresholds[str(group)] = float(benign_threshold(val_scores[val_groups == group], TARGET_FPR))

calibration = {'target_fpr': TARGET_FPR, 'global_threshold': float(GLOBAL_THRESHOLD),
               'min_port_validation': MIN_PORT_VALIDATION, 'min_group_validation': MIN_GROUP_VALIDATION,
               'port_thresholds': port_thresholds, 'group_thresholds': group_thresholds,
               'port_counts': port_counts, 'group_counts': group_counts, 'threshold_policy': 'port -> service group -> global'}
print('Global:', GLOBAL_THRESHOLD, '| eligible ports:', len(port_thresholds), '| eligible groups:', len(group_thresholds))
print('Port 21 validation count:', port_counts.get('21', 0), '| port 8080:', port_counts.get('8080', 0))
print('Port thresholds:', {p:port_thresholds.get(p,'fallback') for p in ['21','8080']})


Global: 0.017142509448652452 | eligible ports: 7 | eligible groups: 3
Port 21 validation count: 361 | port 8080: 201
Port thresholds: {'21': 'fallback', '8080': 'fallback'}


## 5. Save V15 models and calibrated thresholds


In [5]:
selection = {'version': VERSION, 'model_files': [f'autoencoder_{VERSION}_model_seed{s}.keras' for s in ENSEMBLE_SEEDS],
             'target_fpr': TARGET_FPR, 'threshold': float(GLOBAL_THRESHOLD),
             'threshold_source': 'BENIGN validation only', 'calibration_file': f'autoencoder_{VERSION}_local_thresholds.json'}
for model, filename in zip(models, selection['model_files']):
    model.save(MODEL_DIR / filename)
joblib.dump(prep, MODEL_DIR / f'autoencoder_{VERSION}_preprocessing.joblib')
(MODEL_DIR / f'autoencoder_{VERSION}_selection.json').write_text(json.dumps(selection, indent=2))
(MODEL_DIR / selection['calibration_file']).write_text(json.dumps(calibration, indent=2))
print('Saved V15 artifacts to:', MODEL_DIR.resolve())


Saved V15 artifacts to: C:\Users\saaral\Desktop\gc\pattern recog\project\]code\Hybrid-Network-Intrusion-Detection\saved-models\autoencoder_v15
